In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings("ignore")

In [2]:
# 1. Zero-Safe Metric Functions
def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + 1e-8) * 100

def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    return np.mean(np.abs(y_true - y_pred) / (denominator + 1e-8)) * 100

In [3]:
# 2. Load Preprocessed Graph and Tensors
GRAPH_PATH = Path(r"C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\data\processed\intersection_graph.npz")
print(f"Loading graph and tensors from {GRAPH_PATH}...")
npz = np.load(GRAPH_PATH)

adj = npz["adj"]
X_train = npz["X_train"]  # [Samples, T=4, N=3, C=2]
y_train = npz["y_train"]  # [Samples, N=3]
X_test  = npz["X_test"]
y_test  = npz["y_test"]

num_nodes = adj.shape[0]

Loading graph and tensors from C:\Users\KARTIK\OneDrive\Desktop\Master folder\Traffic flow project\data\processed\intersection_graph.npz...


In [4]:
# 3. Compute Normalized Graph Laplacian: D^(-1/2) * A * D^(-1/2)
rowsum = np.sum(adj, axis=1)
d_inv_sqrt = np.power(rowsum, -0.5).flatten()
d_inv_sqrt[np.isinf(d_inv_sqrt)] = 0.0
d_mat_inv_sqrt = np.diag(d_inv_sqrt)
norm_adj = d_mat_inv_sqrt.dot(adj).dot(d_mat_inv_sqrt)
norm_adj_t = torch.FloatTensor(norm_adj)

In [5]:
# 4. Prepare PyTorch Tensors and DataLoaders
X_train_t = torch.FloatTensor(X_train)
y_train_t = torch.FloatTensor(y_train)
X_test_t  = torch.FloatTensor(X_test)
y_test_t  = torch.FloatTensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)

In [6]:
# 5. Define Graph Convolution Layer
class SpatialGraphConv(nn.Module):
    def __init__(self, in_features, out_features):
        super(SpatialGraphConv, self).__init__()
        self.weight = nn.Parameter(torch.FloatTensor(in_features, out_features))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x, adj_matrix):
        # x shape: [Batch, Nodes, in_features]
        # Message passing: A_hat * X * W
        support = torch.matmul(x, self.weight)
        output = torch.matmul(adj_matrix, support)
        return output

In [7]:
# 6. Define Complete STGCN Model
class STGCN(nn.Module):
    def __init__(self, num_nodes, in_channels=2, spatial_hidden=32, temporal_hidden=64):
        super(STGCN, self).__init__()
        self.num_nodes = num_nodes
        
        # Spatial Graph Convolution Layer
        self.gcn = SpatialGraphConv(in_channels, spatial_hidden)
        self.relu = nn.ReLU()
        
        # Temporal Modeling Layer (GRU across time steps)
        self.temporal_gru = nn.GRU(
            input_size=spatial_hidden,
            hidden_size=temporal_hidden,
            num_layers=1,
            batch_first=True
        )
        
        # Multi-node output regressor
        self.fc = nn.Linear(temporal_hidden, 1)

    def forward(self, x, adj_matrix):
        # x shape: [Batch, Time=4, Nodes=3, Channels=2]
        batch_size, time_steps, nodes, channels = x.shape
        
        # Step 1: Apply Spatial Graph Convolution at each time step
        spatial_out = []
        for t in range(time_steps):
            x_t = x[:, t, :, :] # [Batch, Nodes, Channels]
            gcn_t = self.relu(self.gcn(x_t, adj_matrix)) # [Batch, Nodes, spatial_hidden]
            spatial_out.append(gcn_t)
            
        # Shape: [Batch, Time, Nodes, spatial_hidden]
        spatial_seq = torch.stack(spatial_out, dim=1)
        
        # Reshape to treat nodes as parallel sequence instances
        # [Batch * Nodes, Time, spatial_hidden]
        spatial_seq = spatial_seq.permute(0, 2, 1, 3).contiguous()
        spatial_seq = spatial_seq.view(batch_size * nodes, time_steps, -1)
        
        # Step 2: Temporal GRU
        gru_out, _ = self.temporal_gru(spatial_seq)
        last_step = gru_out[:, -1, :] # [Batch * Nodes, temporal_hidden]
        
        # Step 3: Fully connected projection
        out = self.fc(last_step) # [Batch * Nodes, 1]
        out = out.view(batch_size, nodes) # [Batch, Nodes=3]
        return out

In [13]:
# 8. Evaluation
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t.to(device), norm_adj_t).cpu().numpy()
    test_actuals = y_test

# Evaluation on Node 0 (Approach 1) for direct comparison with Ridge, XGBoost, and LSTM
app1_preds = test_preds[:, 0]
app1_actuals = test_actuals[:, 0]

mae_1 = mean_absolute_error(app1_actuals, app1_preds)
rmse_1 = np.sqrt(mean_squared_error(app1_actuals, app1_preds))
wmape_1 = wmape(app1_actuals, app1_preds)
smape_1 = smape(app1_actuals, app1_preds)

# Network-wide Evaluation (average across all 3 approaches simultaneously)
mae_all = mean_absolute_error(test_actuals, test_preds)
rmse_all = np.sqrt(mean_squared_error(test_actuals, test_preds))
wmape_all = wmape(test_actuals, test_preds)
smape_all = smape(test_actuals, test_preds)

print("\n--- STGCN Evaluation Results (Stage 4) ---")
print(f"Approach 1 (Node 0) Benchmark:")
print(f"  MAE       : {mae_1:.4f}")
print(f"  RMSE      : {rmse_1:.4f}")
print(f"  WMAPE (%) : {wmape_1:.2f}%")
print(f"  sMAPE (%) : {smape_1:.2f}%")

print(f"\nAll-Intersection Network Average (Approaches 1, 2, & 3):")
print(f"  MAE       : {mae_all:.4f}")
print(f"  RMSE      : {rmse_all:.4f}")
print(f"  WMAPE (%) : {wmape_all:.2f}%")
print(f"  sMAPE (%) : {smape_all:.2f}%")


--- STGCN Evaluation Results (Stage 4) ---
Approach 1 (Node 0) Benchmark:
  MAE       : 0.0921
  RMSE      : 0.1191
  WMAPE (%) : 15.50%
  sMAPE (%) : 20.85%

All-Intersection Network Average (Approaches 1, 2, & 3):
  MAE       : 0.0851
  RMSE      : 0.1157
  WMAPE (%) : 15.54%
  sMAPE (%) : 22.68%


In [8]:
# 7. Model Training Loop
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = STGCN(num_nodes=num_nodes, in_channels=2, spatial_hidden=32, temporal_hidden=64).to(device)
norm_adj_t = norm_adj_t.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.003)

print(f"Training STGCN on {len(X_train)} samples across {num_nodes} approaches...")
epochs = 25
model.train()
for epoch in range(epochs):
    total_loss = 0.0
    for batch_X, batch_y in train_loader:
        batch_X = batch_X.to(device)
        batch_y = batch_y.to(device)
        
        optimizer.zero_grad()
        preds = model(batch_X, norm_adj_t)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    if (epoch + 1) % 5 == 0:
        print(f"Epoch [{epoch+1}/{epochs}] - Loss: {total_loss / len(train_loader):.4f}")

Training STGCN on 1677 samples across 3 approaches...
Epoch [5/25] - Loss: 0.0175
Epoch [10/25] - Loss: 0.0174
Epoch [15/25] - Loss: 0.0173
Epoch [20/25] - Loss: 0.0169
Epoch [25/25] - Loss: 0.0168


In [ ]:
# 8. Evaluation
# model.eval()
# with torch.no_grad():
#     test_preds = model(X_test_t.to(device), norm_adj_t).cpu().numpy()
#     test_actuals = y_test

In [11]:
# 8. Evaluation
model.eval()
with torch.no_grad():
    test_preds = model(X_test_t.to(device), norm_adj_t).cpu().numpy()
    test_actuals = y_test

# Evaluation on Node 0 (Approach 1) for direct comparison with Ridge, XGBoost, and LSTM
app1_preds = test_preds[:, 0]
app1_actuals = test_actuals[:, 0]

mae_1 = mean_absolute_error(app1_actuals, app1_preds)
rmse_1 = np.sqrt(mean_squared_error(app1_actuals, app1_preds))
wmape_1 = wmape(app1_actuals, app1_preds)
smape_1 = smape(app1_actuals, app1_preds)

# Network-wide Evaluation (average across all 3 approaches simultaneously)
mae_all = mean_absolute_error(test_actuals, test_preds)
rmse_all = np.sqrt(mean_squared_error(test_actuals, test_preds))
wmape_all = wmape(test_actuals, test_preds)
smape_all = smape(test_actuals, test_preds)

print("\n--- STGCN Evaluation Results (Stage 4) ---")
print(f"Approach 1 (Node 0) Benchmark:")
print(f"  MAE       : {mae_1:.4f}")
print(f"  RMSE      : {rmse_1:.4f}")
print(f"  WMAPE (%) : {wmape_1:.2f}%")
print(f"  sMAPE (%) : {smape_1:.2f}%")

print(f"\nAll-Intersection Network Average (Approaches 1, 2, & 3):")
print(f"  MAE       : {mae_all:.4f}")
print(f"  RMSE      : {rmse_all:.4f}")
print(f"  WMAPE (%) : {wmape_all:.2f}%")
print(f"  sMAPE (%) : {smape_all:.2f}%")


--- STGCN Evaluation Results (Stage 4) ---
Approach 1 (Node 0) Benchmark:
  MAE       : 0.0921
  RMSE      : 0.1191
  WMAPE (%) : 15.50%
  sMAPE (%) : 20.85%

All-Intersection Network Average (Approaches 1, 2, & 3):
  MAE       : 0.0851
  RMSE      : 0.1157
  WMAPE (%) : 15.54%
  sMAPE (%) : 22.68%
